# Compare 10

In [1]:
# Compare three proteins to see if the non hydrogen atoms are placed determanistically if the 'seed' parameter is passed to PDBFixer.
# Let's use 1KE9. 
# fixer.addMissingAtoms(seed=26)

In [2]:
prodowndir = "chemfiles/proteindownload/"
ligdowndir = "chemfiles/liganddownload/"
prodir = "chemfiles/protein/"
ligdir = "chemfiles/ligand/"
resultsdir = "results/"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
masterlog = resultsdir + "masterlog.txt"
comparedir = "chemfiles/compare/"

In [3]:
from Bio.PDB import PDBParser, PDBIO, Select

class AtomOnlySelect(Select):
    """Keep all atoms, residues, chains, models—but only if they have ATOM records."""
    def accept_atom(self, atom):
        return True  # Keep all ATOM atoms
    def accept_residue(self, residue):
        return True  # Keep residues with ATOM atoms
    def accept_chain(self, chain):
        return True  # Keep chains with residues
    def accept_model(self, model):
        return True  # Keep models
    def accept_struct(self, struct):
        return True  # Keep structure

def strip_non_atom_pdb(input_path, output_path):
    """
    Parse PDB and write out only ATOM records (no HETATM, CONECT, etc.).

    Parameters
    ----------
    input_path : str
        Input .pdb file.
    output_path : str
        Output .pdb file with only ATOM lines.
    """
    parser = PDBParser(QUIET=True)  # Suppress warnings
    structure = parser.get_structure("stripped", input_path)
    
    io = PDBIO()
    io.set_structure(structure)
    io.save(output_path, AtomOnlySelect())


In [11]:
strip_non_atom_pdb(comparedir + "dwaine_1KE9.pdb", comparedir + "dwaine_1KE9_clean.pdb")
strip_non_atom_pdb(comparedir + "dwaine2_1KE9.pdb", comparedir + "dwaine2_1KE9_clean.pdb")
strip_non_atom_pdb(comparedir + "maya_1KE9.pdb", comparedir + "maya_1KE9_clean.pdb")

In [6]:
from Bio.PDB import PDBParser, PDBIO, Select

class NoHydrogenSelect(Select):
    """Keep only non-hydrogen atoms."""
    def accept_atom(self, atom):
        # Reject atoms where element is 'H' (case-sensitive)
        return atom.element != 'H'
    
    def accept_residue(self, residue):
        return True  # Keep residues even if some atoms removed
    
    def accept_chain(self, chain):
        return True  # Keep chains
    
    def accept_model(self, model):
        return True  # Keep models
    
    def accept_struct(self, struct):
        return True  # Keep structure

def remove_hydrogens(input_path, output_path):
    """
    Remove all hydrogen atoms from PDB file.
    
    Parameters
    ----------
    input_path : str
        Input .pdb file
    output_path : str
        Output .pdb file without hydrogens
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("noH", input_path)
    
    io = PDBIO()
    io.set_structure(structure)
    io.save(output_path, NoHydrogenSelect())


In [12]:
remove_hydrogens(comparedir + "dwaine_1KE9_clean.pdb", comparedir + "dwaine_1KE9_clean_minusH.pdb")
remove_hydrogens(comparedir + "dwaine2_1KE9_clean.pdb", comparedir + "dwaine2_1KE9_clean_minusH.pdb")
remove_hydrogens(comparedir + "maya_1KE9_clean.pdb", comparedir + "maya_1KE9_clean_minusH.pdb")

In [ ]:
from Bio.PDB import PDBParser, PDBIO

def rename_chain(input_path, output_path, old_chain="A", new_chain="B"):
    """
    Rename a chain in a PDB file (e.g., A → B).
    
    Parameters
    ----------
    input_path : str
        Input .pdb file
    output_path : str
        Output .pdb file with renamed chain
    old_chain : str
        Original chain ID (default: "A")
    new_chain : str
        New chain ID (default: "B")
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("renamed", input_path)
    
    # Traverse all models and rename matching chains
    for model in structure:
        for chain in model:
            if chain.get_id() == old_chain:
                print(f"Renaming chain {old_chain} → {new_chain}")
                chain.id = new_chain
    
    # Write modified structure
    io = PDBIO()
    io.set_structure(structure)
    io.save(output_path)


In [ ]:
rename_chain(comparedir + "maya_minusH.pdb", comparedir + "maya_minusH_B.pdb")

In [8]:
from Bio.PDB import PDBParser, Superimposer
import numpy as np

def compare_pdb_coordinates(pdb1_path, pdb2_path, chain_id=None):
    """
    Compare atomic coordinates between two PDB files.
    
    Parameters
    ----------
    pdb1_path, pdb2_path : str
        Paths to input PDB files
    chain_id : str, optional
        Compare only this chain (default: all chains)
    
    Returns
    -------
    rmsd : float
    atom_diffs : list of (atom_name, res_name, res_num, diff_A)
    """
    parser = PDBParser(QUIET=True)
    
    # Parse both structures
    struct1 = parser.get_structure("ref", pdb1_path)
    struct2 = parser.get_structure("target", pdb2_path)
    
    # Extract atoms (optionally filter by chain)
    atoms1, atoms2 = [], []
    
    for model1 in struct1:
        for chain1 in model1:
            if chain_id and chain1.id != chain_id:
                continue
            for residue1 in chain1:
                for atom1 in residue1:
                    atoms1.append(atom1)
    
    for model2 in struct2:
        for chain2 in model2:
            if chain_id and chain2.id != chain_id:
                continue
            for residue2 in chain2:
                for atom2 in residue2:
                    atoms2.append(atom2)
    
    if len(atoms1) != len(atoms2):
        print(f"Warning: Different atom counts ({len(atoms1)} vs {len(atoms2)})")
        return None, []
    
    # Superimpose structures (minimize RMSD)
    sup = Superimposer()
    sup.set_atoms(atoms1, atoms2)
    sup.apply(atoms2)
    
    # Calculate per-atom differences
    coords1 = np.array([a.get_coord() for a in atoms1])
    coords2 = np.array([a.get_coord() for a in atoms2])
    differences = np.linalg.norm(coords2 - coords1, axis=1)
    
    rmsd = np.sqrt(np.mean(differences**2))
    
    # Detailed report (top differences)
    atom_diffs = []
    for i, (atom1, diff) in enumerate(zip(atoms1, differences)):
        if diff > 0.1:  # Report >0.1Å differences
            atom_diffs.append((
                atom1.get_name(),
                atom1.parent.resname,
                atom1.parent.id[1],
                f"{diff:.3f}Å"
            ))
    
    return rmsd, atom_diffs

In [13]:
compare_pdb_coordinates(comparedir + "dwaine_1KE9_clean_minusH.pdb", comparedir + "dwaine2_1KE9_clean_minusH.pdb")

(np.float32(0.0), [])

In [10]:
compare_pdb_coordinates(comparedir + "dwaine_1KE9_clean_minusH.pdb", comparedir + "maya_1KE9_clean_minusH.pdb")

(np.float32(2.2475505),
 [('O', 'GLU', 2, '0.104Å'),
  ('CA', 'ASN', 3, '0.101Å'),
  ('C', 'ASN', 3, '0.102Å'),
  ('O', 'ASN', 3, '0.105Å'),
  ('CB', 'ASN', 3, '0.100Å'),
  ('CG', 'ASN', 3, '0.101Å'),
  ('ND2', 'ASN', 3, '0.103Å'),
  ('CA', 'PHE', 4, '0.100Å'),
  ('C', 'PHE', 4, '0.103Å'),
  ('O', 'PHE', 4, '0.101Å'),
  ('N', 'GLN', 5, '0.107Å'),
  ('CA', 'GLN', 5, '0.110Å'),
  ('C', 'GLN', 5, '0.109Å'),
  ('O', 'GLN', 5, '0.111Å'),
  ('CB', 'GLN', 5, '0.116Å'),
  ('CG', 'GLN', 5, '0.120Å'),
  ('CD', 'GLN', 5, '0.125Å'),
  ('OE1', 'GLN', 5, '0.128Å'),
  ('NE2', 'GLN', 5, '0.126Å'),
  ('N', 'LYS', 6, '0.105Å'),
  ('CA', 'LYS', 6, '0.103Å'),
  ('C', 'LYS', 6, '0.108Å'),
  ('O', 'LYS', 6, '0.111Å'),
  ('N', 'VAL', 7, '0.109Å'),
  ('CA', 'VAL', 7, '0.114Å'),
  ('C', 'VAL', 7, '0.113Å'),
  ('O', 'VAL', 7, '0.116Å'),
  ('CB', 'VAL', 7, '0.115Å'),
  ('CG1', 'VAL', 7, '0.120Å'),
  ('CG2', 'VAL', 7, '0.117Å'),
  ('N', 'GLU', 8, '0.110Å'),
  ('CA', 'GLU', 8, '0.109Å'),
  ('C', 'GLU', 8, '0.104Å'

In [14]:
import nglview as nv
from rdkit import Chem

# From SDF
view = nv.show_structure_file(comparedir + "dwaine_1KE9_clean_minusH.pdb")
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

NGLWidget()

In [19]:
def get_moved_atom_selection(pdb_ref, pdb_sup, threshold=0.2):
    parser = PDBParser(QUIET=True)
    s1 = parser.get_structure("ref", pdb_ref)
    s2 = parser.get_structure("mov", pdb_sup)

    atoms1 = [a for a in s1.get_atoms()]
    atoms2 = [a for a in s2.get_atoms()]

    sel_residues = set()

    for a1, a2 in zip(atoms1, atoms2):
        d = np.linalg.norm(a1.coord - a2.coord)
        if d > threshold:
            res = a1.get_parent()
            chain_id = res.get_parent().id
            resnum = res.id[1]
            sel_residues.add(f"{chain_id}:{resnum}")

    # NGL uses e.g. "A:45 or B:120"
    return " or ".join(sorted(sel_residues))


def view_with_moved_highlight(pdb_ref, pdb_sup, threshold=0.2):
    sel = get_moved_atom_selection(pdb_ref, pdb_sup, threshold)
    print("Moved residues selection:", sel)

    view = nv.NGLWidget()
    view.add_component(pdb_ref)
    view.add_component(pdb_sup)

    # Base cartoons
    view[0].add_cartoon(color="grey")
    view[1].add_cartoon(color="grey")
    view.update_representation(component=1, repr_index=0, opacity=0.3)

    # Highlight moved regions on variant (component 1)
    if sel:
        view[1].add_ball_and_stick(selection=sel, color="red")

    view.center()
    return view

# After superposing:
view2 = view_with_moved_highlight(comparedir + "dwaine_1KE9_clean_minusH.pdb", comparedir + "maya_1KE9_clean_minusH.pdb", threshold=0.2)

view2.camera = 'orthographic'
view2.center()
view2


Moved residues selection: A:150 or A:153 or A:154 or A:155 or A:156 or A:157 or A:158 or A:159 or A:160 or A:161 or A:162 or A:25 or A:36 or A:37 or A:38 or A:39 or A:40 or A:41 or A:42 or A:43


NGLWidget()